In [1]:
!pip install ultralytics
/

()

In [2]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

In [3]:
# Step 1: Install dependencies

!pip install ultralytics opencv-python-headless filterpy lap deep_sort_realtime torch torchvision matplotlib

In [4]:
from google.colab import files

uploaded = files.upload()  # This lets you select a video from your PC
for fn in uploaded.keys():
    input_video_path = fn
    print('Uploaded video:', input_video_path)

Saving input.mp4 to input.mp4
Uploaded video: input.mp4


In [5]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# --------------------------
# Parameters
# --------------------------
input_video_path = "input.mp4"
output_path = "output.mp4"

# --------------------------
# Load YOLOv8 model
# --------------------------
model = YOLO('yolov8n.pt')  # small, fast model

# --------------------------
# Initialize DeepSORT tracker
# --------------------------
tracker = DeepSort(max_age=30)

# --------------------------
# Open video
# --------------------------
cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    print(f"Error: Could not open video file {input_video_path}")
    exit()

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30  # fallback if FPS is 0
print(f"Video opened: {width}x{height} at {fps} FPS")

# --------------------------
# Video writer with MP4V codec (more compatible)
# --------------------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use mp4v codec for broader compatibility
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

if not out.isOpened():
    print(f"Error: Could not open video writer for {output_path}. Try another codec or check permissions.")
    cap.release()
    exit()

# --------------------------
# Counting line setup
# --------------------------
entry_line_y = int(height / 3)
count = 0
frame_idx = 0

print("Starting video processing loop...")
while True:
    ret, frame = cap.read()
    if not ret:
        print(f"End of video or error reading frame at frame {frame_idx}.")
        break

    # Resize frame (just in case)
    frame = cv2.resize(frame, (width, height))

    # YOLO detection
    results = model(frame)
    boxes = results[0].boxes.xyxy
    scores = results[0].boxes.conf

    boxes_np = boxes.cpu().numpy()
    scores_np = scores.cpu().numpy()

    # Prepare detections for DeepSORT
    detections = [[b.tolist(), float(s)] for b, s in zip(boxes_np, scores_np)]

    # DeepSORT tracking
    tracks = tracker.update_tracks(detections, frame=frame)

    # Draw tracks and count objects
    for track in tracks:
        bbox = track.to_ltrb()
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (255,0,0), 2)
        cv2.putText(frame, f'ID:{track.track_id}', (int(bbox[0]), int(bbox[1]-10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

        centroid_y = (bbox[1]+bbox[3])/2
        if centroid_y < entry_line_y and not hasattr(track, 'passed'):
            count += 1
            track.passed = True

    # Draw counting line
    cv2.line(frame, (0, entry_line_y), (width, entry_line_y), (0,255,255), 2)

    # Write frame
    out.write(frame)
    frame_idx += 1

print("Video processing loop finished. Releasing resources...")
# Release resources
cap.release()
out.release()
print(f'Tracking finished! Output saved to {output_path}')
print(f'Total count across line: {count}')

Video opened: 1920x1080 at 26.427920587690412 FPS
Starting video processing loop...

0: 384x640 9 cars, 156.6ms
Speed: 6.9ms preprocess, 156.6ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 139.9ms
Speed: 4.4ms preprocess, 139.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 139.2ms
Speed: 5.2ms preprocess, 139.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 134.6ms
Speed: 4.2ms preprocess, 134.6ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9 cars, 1 bus, 146.5ms
Speed: 5.1ms preprocess, 146.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 10 cars, 1 bus, 147.2ms
Speed: 7.2ms preprocess, 147.2ms inference, 5.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9 cars, 212.1ms
Speed: 4.6ms preprocess, 212.1ms inference, 1.8ms postprocess per 